## Sequence Purchasing Problem

Lead     : `<Alex / AlexLeonardos>`

Issue    : [Github Issue #84](https://github.com/petadex/igem-toronto/issues/) — _Sequence Purchasing Algorithm_

Start    : `2026-06-01`


## Background and Motivation

We want to optimize how we purchase DNA sequences by considering the value of Degenerate-Codon Oligo Libraries. We can purchase a sequence where at specific positions, different nucleotide bases can occur at a ratio we define. Using this, we could encode for many similar sequences from a desired cluster even though we only purchased 1 sequence. Therefore, our motivation is to find the best sequence(s) to purchase for our needs. 

There are several factors that we must account for:

1. **Sequence Information Quantity**: We have a limit in terms of how much information can be ordered, not necessarily the number of sequences. This is only quantified on the total sequences ordered over many clusters (libraries), so it's not computed for this algorithm, which outputs the information used in the optimal solution for specific input clusters (libraries).

2. **Sequence Degeneracy**: How many degenerate bases we will allow in any given sequence.

    - Want to *minimize junk*, critical part of Artem's specification.

3. **Possible Fragment Synthesis Approaches**: Whether ordering multiple smaller sequences and ligating them together could work. This idea is expanded upon in the *Important Operations section*.

4. **Sequence Coverage**: How many natural amino acid sequences (exist in the chosen families) these purchased DNA sequences cover.

5. **Codon Ratios**: Which ratio of bases at these positions would maximize the amount of natural sequences synthesized and minimize the amount of artificial ones that aren't biologically existing.

    - Computed using Lisa's *scoring function*. This considers the codon table for specific vectors which assays would take place using.
    - Max 4 custom ratio bases according to Twist, although we may be able to pay for more
        - IUPAC Standard 50/50 combinations don't count towards this custom ratio count

## Problem Statement

We want to create an algorithm that designs the best nucleotide sequence(s) to order for a given library, with a *junk ratio cap* for how much % junk is acceptable.

### Input: 

   1. **FASTA file for Input**: which consists of *aligned, trimmed protein sequences* that represent unique cores extracted from PETase clusters. The n_k at the end of the lines signifies how many natural sequences mapped to this core.  The fact that they are protein sequences is very important, as the algorithm must have a mechanism of *converting from nucleotides to amino acids*. 
Stored in the form:
```
>core1_n3
AAFAAPAGQTNPYARGPNPTAASLEASAGPFTVRSFTVSRPSGYGAGTVYYPTNAGGTVG
AIAIVPGYTARQSSIKWWGPRLASHGFVVITIDTNSTFDYPSSRSSQQMAALRQVASLNG
DSSSPIYGKVDTARMGVMGHSMGGGASLRSAANNPSLKAAIPQAPWDSQTNFSSVTVPTL
IFACENDSIAPVNSHALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMK
RFMDNDTRYSTFACENPNSTAVSDFRTANCS-
>core2_n2
AVSAAATAQTNPYARGPNPTAASLEASAGPFTVRSFTVSRPSGYGAGTVYYPTNAGGTVG
AIAIVPGYTARQSSIKWWGPRLASHGFVVITIDTNSTLDQPSSRSSQQMAALRQVASLNG
TSSSPIYGKVDTARMGVMGWSMGGGGSLISAANNPSLKAAAPQAPWDSSTNFSSVTVPTL
IFACENDSIAPVNSSALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMK
RFMDNDTRYSTFACENPNSTRVSDFRTANCS-
```
   2. **Junk Ratio Cap**: A scalar value in (0, 1) to determine what the maximum % of junk we would like to allow. This means that the algorithm must have a *manner of keeping track of how many possible proteins could be encoded for by the current solution*, which can be used to compute the junk ratio (how many aren't targets).
   3. **Fragment Joining Method**: This is either Golden Gate or Homologous Recombination. This defines what the junctions would be defined as. Note that due to these techniques and general plasmid design we are under the following hard constraints:

      1. **CGTCTC** and **GAGACG** are BANNED from occuring ANYWHERE in the sequence.
      2. no internal CGGA or GGTG overhangs unless ur doing shared-overhang-minimal-plasmid stuff and want to exclude the first/last fragment(the backbone uses those two overhangs) 


### Output:

   1. The sequence(s) to order - this is in nucleotide bases.
   2. Indication of which positions have degen codons, fragments, and how much junk this generates.

# Important Operations

These are the operations that can be used to design a sequence. The algorithm should consider these in some form when designing the final sequence over an input library.

1. **Degenerate Codons**: When at a certain index, there are several possibilities for nucleotide bases.
2. **Fragments**: Multiple shorter fragments can be ligated together by specific junctions. This allows for consideration of library sequences of different lengths, or for protein differences that can't be efficiently encoded for using degenerate codons.

# Key Definitions:

Let $L_T$ be our target library of cores. This is a weighted set, with each $s \in L_T$ having a weight $w(s)$. 

*Justification for $w(s)$*: In preprocessing before the algorithm, many of the ORFs contain the exact same core sequence. The weight is defined as the number of natural ORFs that are behind the core $s$. This allows for maximization of natural sequence count, but does not change the number of unique cores ordered. Instead, it allows us to choose which cores to include. This is justifiable, as a core observed in more ORFs is more likely to be valid and not an assembly artifact/outlier.

Let $L_O$ be the protein library encoded by our algorithm's output sequences. This is a cartesian product across fragments and degenerate codons, meaning that it grows quickly with the number of variations added.

1. **Coverage**: A core $s \in L_T$ is *covered* if $s \in L_O$. 
   Therefore, the total coverage is defined as $\frac{\sum_{s \in L_O \cap L_T} w(s)}{\sum_{s \in L_T} w(s)}$. This is weighted by $w(s)$.

2. **Junk**: This loosely means useless sequences. This can be split into two different type of junk, minimized at different points in the algorithm.

   a. **Counting Junk**: This is $1 - \frac{|L_T \cap L_O|}{|L_O|}$. In English, this is the proportion of our output library that's *not* covering a sequence in the target library. This is the junk that represent the cap in the initial stage of the algorithm. Note that this junk cap is not a quality metric, it is a feasibility constraint. It is also not weighted, as the entries of $L_O$ do not have inherent weights, which are relevant for coverage.

   b. **Mass Junk**: This is the probability mass of landing on non-targets. This can be influenced by *custom degenerate codon ratios*, which are a future direction for optimizing in specific species. Custom ratios cannot change $L_O$, only the distribution over it — so mass junk is optimized on the support fixed by stage 1, leaving coverage and counting junk unchanged. Note that counting junk is a special case of mass junk, under the assumption that the output library is uniformly distributed.

## For Future Work:

These are ideas that would be nice to have in a complete, publishable version of this algorithm. For this first pass of sequence purchasing, they do not need to be explicitly considered.

1. Structural Modelling - Accounting for if RNA structures (hairpins) make the sequence infeasible. This was previously a main feature of the algorithm's design, but has since been removed.
2. Proofs about Approximation Ratio?
3. **Custom Ratios**: Which ratio of bases at these positions would maximize the amount of natural sequences synthesized and minimize the amount of artificial ones that aren't biologically existing.

# Current Implementation (August 15th, 2026):

### **Stage 1. Operation Context:**

This stage provides the context for our operations (computes the IUPAC codon table for eventual degenerate codon computation, amino acid set, where Golden-Gate junctions could actually occur, under the provided restrictions). No decisions are made at this stage, it only makes pricing a move in Stage 2 constant time. 


### **Stage 2. Greedy:**

Start with an existing core in the library, specifically the one that covers the most natural sequences. 

Greedy nested loop of choosing another core to add, which minimized junk per newly encoded natural sequence.

- The outermost loop loops over $K$, which determines how many fragments to include. Per iteration, $place\_cuts(K)$ is called, which returns a segmentation found through backtracking DFS.
- The next loop checks if a core can still be added. If so, loop over the cores not yet covered.
- In this innermost loop, compute price for the different degen codon additions the current base sequence would need to include to include that core.

This considers adding the new core through degenerate codons or fragments, by computing the price of the possibilities and computing junk. As previously mentioned, the scoring function is junk per newly encoded natural sequence count, which is minimized. Ties are broken by choosing the option that adds less nucleotides.

Stop when no core can be added without going over the junk cap. This computation is done over amino acids, but a reverse mapping is done to check for forbidden recognition sites through a sliding window check.

### **Stage 3. Materialize:**

Turn the design into a concrete set of DNA fragments to order. Check again that no forbidden recognition sites occur.

Output the proper metrics and the variation introduced, over the different K values. Choose the one with the best balance sequence number-per-kilobase.


## Key Current Approximation:

The fact that cuts are fixed before core selection in the greedy loop is a key limitation of the algorithm. This was done to make the search space easier to iterate over, even if it restricts some possibilities. Future work could go into how to augment this architecture, so that the cuts aren't chosen with the assumption that they're all kept.

# Runtime:

This will be a short cell as the full runtime justification would be very lengthy. However, it can be observed through the loop structure that the final algorithm cubic in the number of cores, linear in alignment width. This is an improvement over the exponential search space.

# Example: 

This cell is an LLM-generated test case that will be replaced by actual testing on the PETadex in the future. This is just justification that the algorithm works on a synthetic example.

## Smoke Test Results

Run on the two iSPETase clusters, Golden Gate chemistry, junk cap 80%:

```bash
python unified_design.py ../../ninetypidorfs/cluster2.core.aln.fasta --chemistry gg --k-max 6
```

**cluster2** — 41 unique cores (153 natural sequences), 290 alignment columns:

```
   K      cores     nat seqs      library    junk%   oligos        nt   seq/kb
   1   41/41      153/153              41     0.0%       37    28,410     5.39
   2   31/41      143/153             154    79.9%       26    12,801    11.17
   3   20/41      132/153             100    80.0%       19     5,346    24.69  <== recommended
   4   17/41       81/153              65    73.8%       20     3,915    20.69
   5   17/41       81/153              65    73.8%       21     3,915    20.69
   6   17/41       81/153              65    73.8%       22     3,483    23.26
```

**cluster1** — 62 unique cores (66 natural sequences), 272 alignment columns:

```
   K      cores     nat seqs      library    junk%   oligos        nt   seq/kb
   1   62/62       66/66               62     0.0%       58    46,623     1.42
   2   37/62       41/66              165    77.6%       34    21,924     1.87
   3   25/62       29/66              114    78.1%       23     9,945     2.92
   4   25/62       29/66              114    78.1%       24     9,519     3.05
   5   25/62       29/66              114    78.1%       25     9,324     3.11
   6    6/62        8/66               18    66.7%       13     2,163     3.70  <== recommended
```

**What these confirm:**

1. **The junk cap is honoured in every design** — nothing exceeds 80%, and most designs sit just under it, which is what a feasibility constraint should look like.
2. **Degenerate codons are used, and here they are free.** At $K = 1$, cluster2 encodes all 41 cores with 37 oligos and cluster1 all 62 with 58 — both at 0% junk. Those are widen moves covering two cores with one oligo wherever they differ at a single codon-expressible position.
3. **The Golden Gate designs are buildable as emitted.** Every junction overhang is non-palindromic, GC-balanced, mutually orthogonal, and distinct from the reserved backbone overhangs, and the assembled full-length examples contain no forbidden recognition site.
4. **The recommender rule is the weak point.** Sequences-per-kilobase is a ratio, so it rewards small orders: on cluster1 it recommends $K = 6$, which encodes only 8 of 66 natural sequences. Until that rule is settled, the frontier table is the honest output and the single recommendation should not be read as a decision.
